# 03 · Coastal Inundation Simulation

Multi-scenario storm-surge modelling for Randwick LGA.

> **Prerequisite:** `00_setup`

In [ ]:
import sys, os
if '/content/geo_sp_project/src' not in sys.path:
    sys.path.extend(['/content/geo_sp_project/src', '/content/geo_sp_project/configs'])
    import config
    from geo_sp.auth import init_ee
    init_ee(config.EE_PROJECT)
    print('✅ EE ready')


In [ ]:
import config
from geo_sp.boundaries import get_lga, get_lga_info
from geo_sp.terrain import load_terrain_layers
from geo_sp.coastal_inundation import (
    compute_inundation, query_osm_roads, query_osm_buildings,
    asset_impact, build_inundation_map, export_inundation_to_drive
)

randwick = get_lga(config.LGA_NAME, config.LGA_LAYER)
info     = get_lga_info(randwick)
layers   = load_terrain_layers()
print(f'Area: {info["area_km2"]:.2f} km²')


In [ ]:
# Compute inundation for all scenarios
results = compute_inundation(
    layers['elevation'], randwick,
    config.INUNDATION_SCENARIOS,
    info['area_km2']
)


In [ ]:
# Fetch OSM assets
bounds_coords = randwick.bounds().getInfo()['coordinates'][0]
bbox = [
    min(c[0] for c in bounds_coords), min(c[1] for c in bounds_coords),
    max(c[0] for c in bounds_coords), max(c[1] for c in bounds_coords),
]
df_roads     = query_osm_roads(bbox)
df_buildings = query_osm_buildings(bbox)
print(f'Roads: {len(df_roads)}  |  Buildings: {len(df_buildings)}')


In [ ]:
df_impact = asset_impact(results, df_buildings, df_roads, config.INUNDATION_SCENARIOS)
print(df_impact.to_string(index=False))


In [ ]:
m = build_inundation_map(results, randwick, df_buildings, config.MAP_CENTER)
m.save('inundation_map.html')
from IPython.display import IFrame
IFrame(src='inundation_map.html', width=900, height=550)


In [ ]:
from datetime import datetime
df_impact.to_csv(f'coastal_impact_{datetime.now().strftime("%Y%m%d")}.csv', index=False)
print('Saved CSV')
# Uncomment to export GeoTIFFs to Google Drive:
# export_inundation_to_drive(results, randwick)
